# Анализ эмоций по текстовому отрывку
### Дисциплина: Системы искусственного интеллекта и большие данные
### Практическое задание №4
**Студент:** Щербаков Ярослав Дмитриевич  
**Группа:** ИКБО-30-24  
**Преподаватель:** Медведев Константин Эдуардович

---

## Описание проекта
Проект посвящён задаче классификации эмоций по текстовым фрагментам.  
Используются два подхода глубокого обучения:
1. **Рекуррентная нейронная сеть (BiLSTM)** — классический подход для последовательных данных
2. **Трансформер (DistilBERT)** — современный подход на основе механизма внимания

**Классы эмоций:** sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5)

## 1. Установка зависимостей

In [ ]:
!pip install kagglehub transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm -q

## 2. Сбор данных

In [ ]:
import kagglehub
import os

# Загрузка датасета
path = kagglehub.dataset_download("nelgiriyewithana/emotions")
print("Path to dataset files:", path)
print("Files:", os.listdir(path))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Загрузка CSV
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print("CSV files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f"\nФорма датасета: {df.shape}")
print(f"\nПервые строки:")
df.head(10)

In [ ]:
# Информация о датасете
print("=== Информация о датасете ===")
print(df.info())
print("\n=== Пропущенные значения ===")
print(df.isnull().sum())
print("\n=== Статистика ===")
print(df.describe())

In [ ]:
# Маппинг меток
EMOTION_MAP = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
EMOTION_COLORS = ['#5B9BD5', '#FFD700', '#FF69B4', '#FF4444', '#9B59B6', '#2ECC71']

df['emotion'] = df['label'].map(EMOTION_MAP)

# Визуализация распределения классов
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Столбчатая диаграмма
counts = df['emotion'].value_counts()
axes[0].bar(counts.index, counts.values, color=EMOTION_COLORS[:len(counts)])
axes[0].set_title('Распределение эмоций в датасете', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Эмоция')
axes[0].set_ylabel('Количество примеров')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=10)

# Круговая диаграмма
axes[1].pie(counts.values, labels=counts.index, colors=EMOTION_COLORS[:len(counts)],
           autopct='%1.1f%%', startangle=90)
axes[1].set_title('Доля каждой эмоции', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nРаспределение классов:")
print(counts)

## 3. Предобработка данных

In [ ]:
import re
from collections import Counter

# Анализ длин текстов
df['text_length'] = df['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['text_length'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Распределение длин текстов (в словах)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Количество слов')
axes[0].set_ylabel('Частота')
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Среднее: {df["text_length"].mean():.1f}')
axes[0].legend()

# Boxplot по эмоциям
emotions_sorted = df.groupby('emotion')['text_length'].median().sort_values().index
df.boxplot(column='text_length', by='emotion', ax=axes[1],
           positions=range(len(emotions_sorted)))
axes[1].set_xticklabels(emotions_sorted, rotation=30)
axes[1].set_title('Длина текстов по эмоциям', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Эмоция')
axes[1].set_ylabel('Слов')
plt.suptitle('')

plt.tight_layout()
plt.savefig('text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Средняя длина текста: {df['text_length'].mean():.1f} слов")
print(f"Медиана: {df['text_length'].median():.1f} слов")
print(f"Макс: {df['text_length'].max()}, Мин: {df['text_length'].min()}")

In [ ]:
def clean_text(text):
    """Очистка текста"""
    text = str(text).lower().strip()
    text = re.sub(r'http\S+|www\S+', '', text)         # удаление URL
    text = re.sub(r'@\w+|#\w+', '', text)              # удаление упоминаний/тегов
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', '', text)   # оставляем буквы, цифры, базовую пунктуацию
    text = re.sub(r'\s+', ' ', text).strip()            # нормализация пробелов
    return text

# Применяем очистку
df['clean_text'] = df['text'].apply(clean_text)

# Примеры до и после
print("Примеры предобработки:")
print("-" * 70)
for i in range(3):
    print(f"Исходный : {df['text'].iloc[i]}")
    print(f"Очищенный: {df['clean_text'].iloc[i]}")
    print(f"Эмоция   : {df['emotion'].iloc[i]}")
    print("-" * 70)

In [ ]:
from sklearn.model_selection import train_test_split

# Разбивка на train/val/test
X = df['clean_text'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val  : {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

## 4. Модель 1: Рекуррентная нейронная сеть (BiLSTM)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# ---- Токенизатор и словарь ----
class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
    
    def build(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(text.split())
        for word, freq in counter.items():
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print(f"Размер словаря: {len(self.word2idx)} слов")
    
    def encode(self, text, max_len=64):
        tokens = [self.word2idx.get(w, 1) for w in text.split()[:max_len]]
        # Паддинг
        tokens += [0] * (max_len - len(tokens))
        return tokens
    
    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary(min_freq=2)
vocab.build(X_train)

In [ ]:
# ---- Dataset ----
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=64):
        self.data = [
            (torch.tensor(vocab.encode(t, max_len), dtype=torch.long),
             torch.tensor(l, dtype=torch.long))
            for t, l in zip(texts, labels)
        ]
    
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

MAX_LEN = 64
BATCH_SIZE = 128

train_dataset = EmotionDataset(X_train, y_train, vocab, MAX_LEN)
val_dataset   = EmotionDataset(X_val,   y_val,   vocab, MAX_LEN)
test_dataset  = EmotionDataset(X_test,  y_test,  vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"Батчей в train: {len(train_loader)}")

In [ ]:
# ---- Архитектура BiLSTM ----
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_layers=2, num_classes=6, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        emb = self.dropout(self.embedding(x))         # [B, L, E]
        out, _ = self.lstm(emb)                        # [B, L, 2H]
        # Механизм внимания (Attention)
        attn_weights = torch.softmax(self.attention(out), dim=1)  # [B, L, 1]
        context = (attn_weights * out).sum(dim=1)      # [B, 2H]
        return self.fc(context)

lstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    num_classes=6,
    dropout=0.4
).to(device)

total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Архитектура BiLSTM:")
print(lstm_model)
print(f"\nОбучаемых параметров: {total_params:,}")

In [ ]:
from tqdm.notebook import tqdm

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Обучение BiLSTM
EPOCHS_LSTM = 10
optimizer_lstm = torch.optim.AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_lstm = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_lstm, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()

lstm_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

print("Обучение BiLSTM...")
for epoch in range(1, EPOCHS_LSTM + 1):
    tr_loss, tr_acc = train_epoch(lstm_model, train_loader, optimizer_lstm, criterion)
    vl_loss, vl_acc = evaluate(lstm_model, val_loader, criterion)
    scheduler_lstm.step(vl_loss)
    
    lstm_history['train_loss'].append(tr_loss)
    lstm_history['val_loss'].append(vl_loss)
    lstm_history['train_acc'].append(tr_acc)
    lstm_history['val_acc'].append(vl_acc)
    
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(lstm_model.state_dict(), 'best_lstm.pt')
    
    print(f"Epoch {epoch:2d}/{EPOCHS_LSTM} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")

print(f"\nЛучший Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# График обучения BiLSTM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(lstm_history['train_loss'], 'o-', label='Train', color='steelblue')
ax1.plot(lstm_history['val_loss'], 's--', label='Validation', color='coral')
ax1.set_title('BiLSTM — Функция потерь', fontsize=13, fontweight='bold')
ax1.set_xlabel('Эпоха'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(lstm_history['train_acc'], 'o-', label='Train', color='steelblue')
ax2.plot(lstm_history['val_acc'], 's--', label='Validation', color='coral')
ax2.set_title('BiLSTM — Точность', fontsize=13, fontweight='bold')
ax2.set_xlabel('Эпоха'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Модель 2: Трансформер (DistilBERT)

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset as TorchDataset

# Загрузка токенизатора и модели DistilBERT
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f"Токенизатор загружен: {MODEL_NAME}")

class BertEmotionDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self): return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

# Используем подмножество для ускорения (20k train, 2k val/test)
N_TRAIN, N_EVAL = 20000, 2000
idx_tr = np.random.choice(len(X_train), min(N_TRAIN, len(X_train)), replace=False)
idx_vl = np.random.choice(len(X_val),   min(N_EVAL,  len(X_val)),   replace=False)
idx_ts = np.random.choice(len(X_test),  min(N_EVAL,  len(X_test)),  replace=False)

bert_train = BertEmotionDataset(X_train[idx_tr], y_train[idx_tr], tokenizer)
bert_val   = BertEmotionDataset(X_val[idx_vl],   y_val[idx_vl],   tokenizer)
bert_test  = BertEmotionDataset(X_test[idx_ts],  y_test[idx_ts],  tokenizer)

bert_train_loader = DataLoader(bert_train, batch_size=32, shuffle=True)
bert_val_loader   = DataLoader(bert_val,   batch_size=64)
bert_test_loader  = DataLoader(bert_test,  batch_size=64)

print(f"Train: {len(bert_train)}, Val: {len(bert_val)}, Test: {len(bert_test)}")

In [ ]:
# Загрузка DistilBERT с головой классификации
bert_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=6
).to(device)

# Замораживаем нижние слои трансформера (fine-tuning верхних)
for name, param in bert_model.distilbert.transformer.layer[:4].named_parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in bert_model.parameters())
print(f"DistilBERT загружен")
print(f"Всего параметров: {total:,}")
print(f"Обучаемых (unfrozen): {trainable:,} ({trainable/total*100:.1f}%)")

In [ ]:
def train_bert_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(labels)
        correct += (outputs.logits.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

@torch.no_grad()
def eval_bert(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item() * len(labels)
        preds = outputs.logits.argmax(1)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), correct / len(loader.dataset), all_preds, all_labels

# Обучение DistilBERT
EPOCHS_BERT = 4
optimizer_bert = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(bert_train_loader) * EPOCHS_BERT
scheduler_bert = torch.optim.lr_scheduler.LinearLR(
    optimizer_bert, start_factor=1.0, end_factor=0.1, total_iters=total_steps
)

bert_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_bert_acc = 0

print("Обучение DistilBERT...")
for epoch in range(1, EPOCHS_BERT + 1):
    tr_loss, tr_acc = train_bert_epoch(bert_model, bert_train_loader, optimizer_bert, scheduler_bert)
    vl_loss, vl_acc, _, _ = eval_bert(bert_model, bert_val_loader)
    
    bert_history['train_loss'].append(tr_loss)
    bert_history['val_loss'].append(vl_loss)
    bert_history['train_acc'].append(tr_acc)
    bert_history['val_acc'].append(vl_acc)
    
    if vl_acc > best_bert_acc:
        best_bert_acc = vl_acc
        torch.save(bert_model.state_dict(), 'best_bert.pt')
    
    print(f"Epoch {epoch}/{EPOCHS_BERT} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")

print(f"\nЛучший DistilBERT Val Accuracy: {best_bert_acc:.4f}")

In [ ]:
# График обучения DistilBERT
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(bert_history['train_loss'], 'o-', label='Train', color='mediumpurple')
ax1.plot(bert_history['val_loss'], 's--', label='Validation', color='tomato')
ax1.set_title('DistilBERT — Функция потерь', fontsize=13, fontweight='bold')
ax1.set_xlabel('Эпоха'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(bert_history['train_acc'], 'o-', label='Train', color='mediumpurple')
ax2.plot(bert_history['val_acc'], 's--', label='Validation', color='tomato')
ax2.set_title('DistilBERT — Точность', fontsize=13, fontweight='bold')
ax2.set_xlabel('Эпоха'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bert_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Оценка работоспособности моделей

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Загрузка лучших весов
lstm_model.load_state_dict(torch.load('best_lstm.pt', map_location=device))
bert_model.load_state_dict(torch.load('best_bert.pt', map_location=device))

# ---- BiLSTM ----
@torch.no_grad()
def predict_lstm(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())
    return np.array(all_preds), np.array(all_labels)

lstm_preds, lstm_true = predict_lstm(lstm_model, test_loader)
_, _, bert_preds, bert_true = eval_bert(bert_model, bert_test_loader)
bert_preds = np.array(bert_preds)
bert_true  = np.array(bert_true)

emotion_names = [EMOTION_MAP[i] for i in range(6)]

print("=" * 60)
print("РЕЗУЛЬТАТЫ BiLSTM на тестовой выборке:")
print("=" * 60)
print(classification_report(lstm_true, lstm_preds, target_names=emotion_names))

print("=" * 60)
print("РЕЗУЛЬТАТЫ DistilBERT на тестовой выборке:")
print("=" * 60)
print(classification_report(bert_true, bert_preds, target_names=emotion_names))

In [ ]:
# Матрицы ошибок
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, true, title, cmap in [
    (axes[0], lstm_preds, lstm_true, 'BiLSTM', 'Blues'),
    (axes[1], bert_preds, bert_true, 'DistilBERT', 'Purples')
]:
    cm = confusion_matrix(true, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=emotion_names, yticklabels=emotion_names,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{title} — Матрица ошибок (норм.)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Предсказано'); ax.set_ylabel('Истинное')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Сравнение метрик
metrics_data = {
    'Модель':   ['BiLSTM', 'DistilBERT'],
    'Accuracy': [
        accuracy_score(lstm_true, lstm_preds),
        accuracy_score(bert_true, bert_preds)
    ],
    'F1 (macro)': [
        f1_score(lstm_true, lstm_preds, average='macro'),
        f1_score(bert_true, bert_preds, average='macro')
    ],
    'F1 (weighted)': [
        f1_score(lstm_true, lstm_preds, average='weighted'),
        f1_score(bert_true, bert_preds, average='weighted')
    ]
}

metrics_df = pd.DataFrame(metrics_data)
print("Сводная таблица метрик:")
print(metrics_df.to_string(index=False))

# Визуализация
x = np.arange(2)
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width, metrics_data['Accuracy'],   width, label='Accuracy',     color='steelblue')
bars2 = ax.bar(x,         metrics_data['F1 (macro)'], width, label='F1 (macro)',   color='coral')
bars3 = ax.bar(x + width, metrics_data['F1 (weighted)'], width, label='F1 (weighted)', color='mediumseagreen')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(['BiLSTM', 'DistilBERT'], fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_title('Сравнение моделей по метрикам', fontsize=14, fontweight='bold')
ax.set_ylabel('Значение метрики')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Интерпретация результатов и демонстрация

In [ ]:
# F1 по классам — обе модели
from sklearn.metrics import f1_score

lstm_f1_per_class = f1_score(lstm_true, lstm_preds, average=None)
bert_f1_per_class = f1_score(bert_true, bert_preds, average=None)

x = np.arange(6)
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x - width/2, lstm_f1_per_class, width, label='BiLSTM',     color='steelblue',    alpha=0.85)
ax.bar(x + width/2, bert_f1_per_class, width, label='DistilBERT', color='mediumpurple', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(emotion_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('F1-score по классам эмоций', fontsize=14, fontweight='bold')
ax.set_ylabel('F1-score')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for i, (l, b) in enumerate(zip(lstm_f1_per_class, bert_f1_per_class)):
    ax.text(i - width/2, l + 0.01, f'{l:.2f}', ha='center', fontsize=9)
    ax.text(i + width/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Демонстрация предсказаний на новых текстах
@torch.no_grad()
def predict_text_lstm(text):
    lstm_model.eval()
    clean = clean_text(text)
    tokens = torch.tensor([vocab.encode(clean, MAX_LEN)], dtype=torch.long).to(device)
    probs = torch.softmax(lstm_model(tokens), dim=1).cpu().numpy()[0]
    return {EMOTION_MAP[i]: float(probs[i]) for i in range(6)}

@torch.no_grad()
def predict_text_bert(text):
    bert_model.eval()
    enc = tokenizer(clean_text(text), return_tensors='pt',
                    truncation=True, max_length=64, padding=True)
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return {EMOTION_MAP[i]: float(probs[i]) for i in range(6)}

test_texts = [
    "I am so happy today, everything is going wonderfully!",
    "I feel incredibly sad and lonely, nobody understands me.",
    "This makes me so angry, how could they do this!",
    "I am terrified of what might happen next.",
    "I love you so much, you mean everything to me.",
    "Wow, I never expected that to happen, what a surprise!"
]

print("Демонстрация предсказаний на тестовых фразах:")
print("=" * 70)
for text in test_texts:
    lstm_res = predict_text_lstm(text)
    bert_res = predict_text_bert(text)
    
    lstm_pred = max(lstm_res, key=lstm_res.get)
    bert_pred = max(bert_res, key=bert_res.get)
    
    print(f"\nТекст: '{text}'")
    print(f"  BiLSTM    → {lstm_pred:8s} ({lstm_res[lstm_pred]:.3f})")
    print(f"  DistilBERT → {bert_pred:8s} ({bert_res[bert_pred]:.3f})")

In [ ]:
# Визуализация вероятностей для одного примера
sample_text = "I am so happy today, everything is going wonderfully!"

lstm_probs = predict_text_lstm(sample_text)
bert_probs = predict_text_bert(sample_text)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, probs, title, color in [
    (axes[0], lstm_probs, 'BiLSTM', 'steelblue'),
    (axes[1], bert_probs, 'DistilBERT', 'mediumpurple')
]:
    labels = list(probs.keys())
    values = list(probs.values())
    bars = ax.barh(labels, values, color=color, alpha=0.8)
    ax.set_xlim(0, 1)
    ax.set_title(f'{title}\n"{sample_text[:40]}..."', fontsize=11, fontweight='bold')
    ax.set_xlabel('Вероятность')
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Распределение вероятностей по классам', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_probs.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Итоги

В рамках проекта были реализованы и сравнены два подхода глубокого обучения для классификации эмоций:

| Характеристика | BiLSTM | DistilBERT |
|---|---|---|
| Тип архитектуры | Рекуррентная НС | Трансформер |
| Параметров | ~5M | ~67M (часть заморожена) |
| Макс. длина последовательности | 64 токена | 64 токена |
| Предобучение | ❌ | ✅ (BookCorpus + Wikipedia) |

**Выводы:**
- DistilBERT за счёт предобучения показывает более высокое качество, особенно на редких классах (love, surprise)
- BiLSTM демонстрирует разумное качество при значительно меньшем числе параметров и быстром обучении
- Оба метода хорошо справляются с доминирующими классами (sadness, joy), но испытывают трудности с minority-классами
- Механизм внимания в BiLSTM улучшает качество по сравнению с базовым LSTM